# Cross-Dataset Generalisation — Colab Runner

### Steps
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run **Cell 1** (setup + install)
3. When Cell 1 finishes: **Runtime → Restart runtime**
4. Run **Cells 2 → 3 → 4 → 5 → 6 → 7** in order. Skip Cell 1.

Estimated time: ~2 hours on T4 GPU

In [ ]:
# CELL 1 — Setup + Install  (run once, then restart)
import os

REPO_URL    = "https://github.com/adnankhalil22/fake-news-generalization.git"
REPO_DIR    = "/content/fake-news-generalization"
PROJECT_DIR = os.path.join(REPO_DIR, "OneDrive", "Desktop", "final research")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR} -q
    print("Cloned repo.")
else:
    !git -C {REPO_DIR} pull -q
    print("Pulled latest changes.")

%cd {PROJECT_DIR}
print("Working dir:", os.getcwd())

# ── Install ───────────────────────────────────────────────────
# Do NOT install torch — Colab already has a compatible version.
# Installing requirements.txt downgrades torch and breaks torchvision.

print("
Installing packages (~2 min)...")
!pip install -q --upgrade transformers accelerate datasets huggingface-hub tokenizers
!pip install -q scikit-learn==1.4.2 seaborn joblib spacy

import subprocess, sys
r = subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
                   capture_output=True, text=True)
print("spaCy:", "OK" if r.returncode == 0 else "failed (only needed for masking, not training)")

print()
print("=" * 55)
print("  RESTART NOW: Runtime > Restart runtime")
print("  Then run Cells 2-7. Do NOT re-run Cell 1.")
print("=" * 55)

In [ ]:
# CELL 2 — Re-enter project directory after restart  (run every session)
import os

PROJECT_DIR = "/content/fake-news-generalization/OneDrive/Desktop/final research"

if not os.path.exists(PROJECT_DIR):
    raise RuntimeError("Project not found. Run Cell 1 first, restart, then continue here.")

%cd {PROJECT_DIR}
print("Working directory:", os.getcwd())
print("Contents:", [f for f in os.listdir(".") if not f.startswith(".")][:10])

In [ ]:
# CELL 3 — Verify GPU and datasets
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
if torch.cuda.is_available():
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("Go to Runtime > Change runtime type > T4 GPU > Save, then restart.")

print()
from src.data import get_dataset, DATASETS
all_ok = True
for ds in DATASETS:
    try:
        df = get_dataset(ds, "train")
        print(f"  OK  {ds:8s}  n={len(df):6d}  fake={df.label.mean():.1%}")
    except Exception as e:
        print(f"  ERR {ds}: {e}")
        all_ok = False

print()
print("Ready to train." if (all_ok and torch.cuda.is_available()) else "Fix issues above first.")

In [ ]:
# CELL 4 — Train DistilBERT  (~40 min per dataset, ~2 hrs total)
# Keep this tab active. Checkpoint saved after each epoch.
# If Colab disconnects, re-run Cells 2+3+4 — already-trained datasets are skipped.
import os
from src.data import DATASETS
from src.train import train_distilbert

SEED = 42
results = {}

for ds in DATASETS:
    model_dir = f"models/distilbert_{ds}_seed{SEED}"
    if os.path.exists(os.path.join(model_dir, "config.json")):
        print(f"
Skipping {ds.upper()} — already trained at {model_dir}")
        results[ds] = "skipped (checkpoint exists)"
        continue

    print(f"
{'='*55}")
    print(f"  Training DistilBERT on {ds.upper()}  (seed={SEED})")
    print(f"{'='*55}")
    try:
        m = train_distilbert(ds, seed=SEED)
        f1  = m.get("eval_macro_f1",  m.get("macro_f1",  "n/a"))
        acc = m.get("eval_accuracy",   m.get("accuracy",  "n/a"))
        print(f"  val macro-F1 : {f1}")
        print(f"  val accuracy : {acc}")
        results[ds] = {"macro_f1": f1, "accuracy": acc}
    except Exception as e:
        import traceback; traceback.print_exc()
        results[ds] = f"FAILED: {e}"

print("
Summary:")
for ds, r in results.items():
    print(f"  {ds}: {r}")

In [ ]:
# CELL 5 — Build 3x3 evaluation matrix and heatmap
from src.evaluate import build_matrix
from IPython.display import Image, display

distilbert_matrix = build_matrix("distilbert", seeds=[42])
display(Image("results/figures/distilbert_f1_heatmap.png"))

In [ ]:
# CELL 6 — Save results to Google Drive
from google.colab import drive
import shutil, os

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/fake-news-generalization-results"
os.makedirs(DEST, exist_ok=True)
shutil.copytree("results", os.path.join(DEST, "results"), dirs_exist_ok=True)
print("Saved to Drive:", DEST)

In [ ]:
# CELL 7 — Print final results  (copy this output and paste to Claude)
import pandas as pd

print("=" * 60)
print("DistilBERT Macro-F1 Matrix (rows=train, cols=test)")
print("=" * 60)
df_bert = pd.read_csv("results/matrices/distilbert_f1_matrix.csv", index_col=0)
print(df_bert.to_string())

print("
" + "=" * 60)
print("LogReg Macro-F1 Matrix (for comparison)")
print("=" * 60)
df_lr = pd.read_csv("results/matrices/logreg_f1_matrix.csv", index_col=0)
print(df_lr.to_string())

print("
" + "=" * 60)
print("Gap analysis (in-domain minus best transfer per row)")
print("=" * 60)
for name, df in [("DistilBERT", df_bert), ("LogReg", df_lr)]:
    print(f"
{name}:")
    for ds in df.index:
        ind  = float(df.loc[ds, ds])
        best = max(float(df.loc[ds, c]) for c in df.columns if c != ds)
        print(f"  train={ds:8s}  in-domain={ind:.4f}  best-transfer={best:.4f}  gap={ind-best:+.4f}")

print("
--- Copy everything above this line and paste to Claude ---")